In [4]:
import json
from uuid import uuid4

import numpy as np
import pandas as pd

from pydantic import BaseModel
from beir.datasets.data_loader import GenericDataLoader

## Generate BEIR Dataset

In [5]:
class BEIRQueryItem(BaseModel):
    search_term: str
    answer: str


class BEIRQuery(BaseModel):
    queries: list[BEIRQueryItem]

In [6]:
BEIR_DATA_ROOT = "../data/beir"

In [7]:
df = pd.read_excel(
    f"../dataset/data-corpus-base.xlsx", sheet_name="BEIR Corpus"
).dropna()
df.head()

,Kode Doc,Kode Dokumen,Judul,Konten
0,doc1,b2044980-d3a0-4cf5-91b7-69cda3b7aafe,Wakil Dubes Walanda Gumbira Ningali Holland In...,bogor hotél institut (bhi) gawé bareng jeung f...
1,doc2,3fe1a195-5081-4855-95fa-cba5eff1cdcf,Warga Bogor Mapag Taun anyar Islam,bogor - datang ton anyar islam 1431-hijréh pap...
2,doc3,c121c9b4-a97a-4ba4-8f5e-5e58578b17ce,Warugan Lemah: Pola Lembur Urang Sunda Buhun,naskah warugan lemah kandelna ngan tilu lempir...
3,doc4,b87199ed-ca2f-4e2b-a323-23f3b387a9aa,Manfaat Olahraga Pikeun Kasehatan,anu ku urang tos terang olahraga teh penting p...
4,doc5,cc6a7052-6620-46cc-8704-9f54e3e99016,DINA JANDÉLA INDUNG,"méméh layung kubur panineungan dina jandéla, g..."


In [8]:
with open(f"{BEIR_DATA_ROOT}/corpus.jsonl", "w") as corpus_file:
    for row in df.itertuples():
        json.dump(
            (
                {
                    "_id": row[2],
                    "title": row.Judul,
                    "text": row.Konten,
                }
            ),
            corpus_file,
        )
        corpus_file.write("\n")

In [9]:
beir_corpus_id_map = {}
with open(f"{BEIR_DATA_ROOT}/beir_map.jsonl", "r") as map_file:
    for line in map_file:
        parsed = json.loads(line)
        beir_corpus_id_map[parsed["custom_id"]] = parsed["doc_id"]

In [10]:
beir_total_tokens = 0

with (
    open(
        f"{BEIR_DATA_ROOT}/batch_676f79b4369081909de9dd17e0ecc666_output.jsonl", "r"
    ) as batch_file,
    open(f"{BEIR_DATA_ROOT}/queries.jsonl", "w") as queries_file,
    open(f"{BEIR_DATA_ROOT}/qrels.tsv", "w") as qrels_file,
):
    # write BEIR qrels header
    qrels_file.write("query-id\tcorpus-id\tscore\n")

    # process each batch
    for line in batch_file:
        parsed = json.loads(line)
        model = BEIRQuery(
            **json.loads(parsed["response"]["body"]["choices"][0]["message"]["content"])
        )
        beir_total_tokens += parsed["response"]["body"]["usage"]["total_tokens"]

        doc_id = beir_corpus_id_map[parsed["custom_id"]]
        for item in model.queries:
            query_id = str(uuid4())

            qrels_file.write(f"{query_id}\t{doc_id}\t1\n")

            json.dump({"_id": query_id, "text": item.search_term}, queries_file)
            queries_file.write("\n")

In [11]:
beir_total_tokens

1140703

In [12]:
corpus, queries, qrels = GenericDataLoader(
    data_folder=BEIR_DATA_ROOT, qrels_file=f"{BEIR_DATA_ROOT}/qrels.tsv"
).load_custom()

100%|██████████| 1582/1582 [00:00<00:00, 156108.43it/s]


## Generate MS-MARCO Triplet

In [13]:
MARCO_DATA_ROOT = "../data/marco"

In [14]:
class MSMARCOTripletItem(BaseModel):
    query: str
    positive_passage: str
    negative_passage: str


class MSMARCO(BaseModel):
    triplets: list[MSMARCOTripletItem]

In [16]:
marco_total_tokens = 0

with (
    open(
        f"{MARCO_DATA_ROOT}/batch_676f79bab1988190b988ae5b09f4ffbf_output.jsonl", "r"
    ) as batch_file,
    open(f"{MARCO_DATA_ROOT}/triplet.jsonl", "w") as triplet_file,
):
    for line in batch_file:
        parsed = json.loads(line)
        model = MSMARCO(
            **json.loads(parsed["response"]["body"]["choices"][0]["message"]["content"])
        )
        marco_total_tokens += parsed["response"]["body"]["usage"]["total_tokens"]

        for item in model.triplets:
            json.dump(
                {
                    "query": item.query,
                    "positive": item.positive_passage,
                    "negative": item.negative_passage,
                },
                triplet_file,
            )
            triplet_file.write("\n")

In [17]:
marco_total_tokens

1477697